# Домашнее задание №4

## Методы детекции характеристических точек

### Цель

Измерить, как детекторы характеристических точек и дескрипторы ведут себя при контролируемых искажениях изображения (поворот, масштаб, освещённость), и определить границы применимости сопоставления по шаблону и по признакам.

Результатом работы является не набор красивых картинок с соответствиями, а обоснованный вывод о том, какой детектор и какая схема фильтрации соответствий пригодны для заданного класса искажений.

[Методические указания блока](README.md) · [Общие МУ](../../../docs/guidelines-students.md) · [Рубрика оценивания](../teachers-assessment/README.md)

## 1. Что используется в работе

Библиотеки: `opencv-python`, `numpy`, `scikit-image`, `matplotlib`, `pandas`.

Данные. Ноутбук работает без интернета: пары изображений формируются синтетически из встроенных изображений `skimage.data` применением известной гомографии. Это принципиально: эталонная гомография известна точно, поэтому повторяемость и качество сопоставления измеряются, а не оцениваются на глаз.

Дополнительно вы можете подключить реальные последовательности с известными гомографиями (карточка Oxford Affine Covariant Regions в [resources/datasets](../../resources/datasets/README.md)). В этом случае матрицы гомографий берутся из файлов `H1to2p` и т. д., а весь остальной код ноутбука не меняется.

Заготовка содержит: генерацию пар с известной гомографией, расчёт повторяемости и ошибки переноса, визуализацию соответствий, журнал экспериментов и каркас сводных таблиц. Реализация детекторов, схем фильтрации и планирование серий — ваша часть.

Ожидаемая структура проекта:

```text
hw4/
├── outputs_hw4/
│   ├── figures/
│   └── runs.csv
└── hw4.ipynb
```

In [ ]:
# Зависимости (при необходимости раскомментируйте):
# %pip install opencv-python numpy scikit-image matplotlib pandas

import platform
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import skimage
from skimage import data

SEED = 42
rng = np.random.default_rng(SEED)
cv2.setRNGSeed(SEED)

OUTPUT_DIR = Path("outputs_hw4")
(OUTPUT_DIR / "figures").mkdir(parents=True, exist_ok=True)

print("python      :", platform.python_version())
print("opencv      :", cv2.__version__)
print("numpy       :", np.__version__)
print("scikit-image:", skimage.__version__)
print("pandas      :", pd.__version__)
print("SEED        :", SEED)

## 2. Краткая теоретическая справка

### 2.1. Угловые детекторы

В окрестности точки рассматривается матрица вторых моментов градиента

$$M(x,y) = \sum_{u,v} w(u,v)
\begin{bmatrix} I_x^2 & I_x I_y \\ I_x I_y & I_y^2 \end{bmatrix},$$

где $I_x, I_y$ — частные производные яркости, $w$ — весовое окно.

Критерий Харриса использует определитель и след матрицы:

$$R = \det(M) - k\,\bigl(\operatorname{tr} M\bigr)^2, \qquad k \in [0.04,\ 0.06].$$

Критерий Ши—Томаси использует минимальное собственное число:

$$R = \min(\lambda_1, \lambda_2).$$

Оба критерия описывают одну и ту же локальную структуру, но по-разному штрафуют вытянутые конфигурации собственных чисел (края).

FAST не вычисляет градиенты: точка считается угловой, если на окружности радиуса 3 существует непрерывная дуга из $n$ пикселей, все из которых ярче $I_p + t$ либо темнее $I_p - t$.

### 2.2. Масштаб и DoG

Разность гауссиан аппроксимирует нормированный лапласиан:

$$D(x,y,\sigma) = \bigl(G(x,y,k\sigma) - G(x,y,\sigma)\bigr) * I(x,y) \approx (k-1)\,\sigma^2 \nabla^2 G * I .$$

Экстремумы $D$ ищутся одновременно по координатам и по $\sigma$, поэтому характерный масштаб точки определяется из данных. Именно это, а не сам дескриптор, делает SIFT устойчивым к изменению масштаба; ориентация по доминирующему направлению градиента добавляет устойчивость к повороту.

### 2.3. Сопоставление дескрипторов и ratio test

Для дескриптора $d$ первого изображения находятся два ближайших соседа $d_1, d_2$ во втором. Соответствие принимается, если

$$\frac{\lVert d - d_1 \rVert}{\lVert d - d_2 \rVert} < \tau, \qquad \tau \approx 0.7\text{–}0.8 .$$

Смысл: отвергаются точки, у которых во втором изображении есть несколько одинаково правдоподобных кандидатов (повторяющаяся текстура). Порог $\tau$ управляет компромиссом «число соответствий — доля верных», и в работе его влияние требуется измерить, а не принять по умолчанию.

Кросс-проверка принимает пару $(i, j)$, только если $j$ — ближайший сосед для $i$ и одновременно $i$ — ближайший сосед для $j$.

RANSAC оценивает гомографию по случайным минимальным выборкам и оставляет соответствия, согласованные с моделью:

$$\lVert \mathbf{x}' - H\mathbf{x} \rVert < \varepsilon .$$

### 2.4. Повторяемость

Пусть $H$ — известная гомография, переводящая координаты первого изображения во второе. Повторяемость при допуске $\varepsilon$:

$$\mathrm{Rep}(\varepsilon) = \frac{\bigl|\{\, i : \min_j \lVert H\mathbf{x}_i - \mathbf{x}'_j \rVert \le \varepsilon \,\}\bigr|}{\bigl|\{\, i : H\mathbf{x}_i \in \Omega' \,\}\bigr|},$$

где $\Omega'$ — область второго кадра. В знаменателе учитываются только точки, эталонное положение которых попадает в кадр: иначе повторяемость искусственно занижается уходом точек за границу.

Повторяемость зависит от числа детектируемых точек. Сравнение детекторов при существенно разном числе точек некорректно: фиксируйте `max_points`.

## 3. Задачи

Формулировка из [методических указаний блока](README.md#дз4-методы-детекции-характеристических-точек):

1. Примените и сравните детекторы точек (Харрис, Ши-Томаси, FAST, блобы/DoG) на изображениях с поворотом, изменением масштаба и освещённости.
2. Сопоставьте изображения одной сцены с разных ракурсов дескрипторами (SIFT/ORB) и отфильтруйте соответствия (ratio test, кросс-проверка, RANSAC).
3. Сравните template matching и feature matching: покажите, где шаблонный метод отказывает.

**Результат:** таблица «детектор × искажение → повторяемость», визуализация соответствий до и после фильтрации, вывод об областях применимости.

Проверяемые элементы ([рубрика](../teachers-assessment/README.md)): повторяемость измерена на контролируемых искажениях, а не «на глаз»; соответствия отфильтрованы минимум двумя способами; показан пример отказа template matching.

## 4. Данные: пары с известной гомографией

Искажение задаётся аффинным преобразованием (поворот вокруг центра и масштаб), которое записывается в матрицу $3\times3$, и фотометрическим преобразованием $I \mapsto gI + b$ с аддитивным шумом. Геометрическая часть известна точно, поэтому эталонное положение любой точки первого кадра во втором вычисляется без разметки.

Важно: фотометрическая часть не меняет координаты, поэтому влияние освещённости и влияние геометрии разделяются — это и есть контролируемая серия из [общих МУ](../../../docs/guidelines-students.md#2-методика-эксперимента).

In [ ]:
def load_base_gray(name: str = "astronaut") -> np.ndarray:
    """Загрузить базовое изображение в оттенках серого.

    Вход:  name — 'astronaut' | 'coffee' | 'camera' | 'checkerboard'.
    Выход: np.ndarray (H, W) uint8.
    """
    source = {
        "astronaut": data.astronaut,
        "coffee": data.coffee,
        "camera": data.camera,
        "checkerboard": data.checkerboard,
    }[name]()
    if source.ndim == 3:
        source = cv2.cvtColor(source, cv2.COLOR_RGB2GRAY)
    return np.ascontiguousarray(source.astype(np.uint8))


def make_pair(image, *, angle=0.0, scale=1.0, gain=1.0, bias=0.0,
              noise_sigma=0.0, seed=SEED):
    """Построить пару «исходное изображение -> искажённое» с известной гомографией.

    Вход:
        image        — (H, W) uint8;
        angle        — поворот вокруг центра, градусы;
        scale        — коэффициент масштаба;
        gain, bias   — фотометрическое преобразование I -> gain * I + bias;
        noise_sigma  — СКО аддитивного гауссова шума в единицах яркости.
    Выход:
        warped — (H, W) uint8;
        H      — (3, 3) float64, переводит однородные координаты исходного
                 изображения в координаты warped.
    """
    h, w = image.shape[:2]
    M = cv2.getRotationMatrix2D((w / 2.0, h / 2.0), angle, scale)
    H = np.vstack([M, [0.0, 0.0, 1.0]]).astype(np.float64)
    warped = cv2.warpAffine(image, M, (w, h), flags=cv2.INTER_LINEAR,
                            borderMode=cv2.BORDER_REFLECT101)
    out = warped.astype(np.float32) * gain + bias
    if noise_sigma > 0:
        out = out + np.random.default_rng(seed).normal(0.0, noise_sigma, out.shape)
    return np.clip(out, 0, 255).astype(np.uint8), H


def warp_points(H, pts):
    """Применить гомографию к точкам.

    Вход:  H (3, 3); pts (N, 2) в формате (x, y).
    Выход: (N, 2) float64.
    """
    pts = np.asarray(pts, dtype=np.float64).reshape(-1, 1, 2)
    return cv2.perspectiveTransform(pts, np.asarray(H, dtype=np.float64)).reshape(-1, 2)


base_gray = load_base_gray("astronaut")
print("Базовое изображение:", base_gray.shape, base_gray.dtype)

In [ ]:
# Набор контролируемых искажений. Каждая строка меняет ОДИН фактор
# относительно identity; смешанные искажения добавляйте отдельной серией
# и помечайте это в журнале.
DISTORTIONS = {
    "identity":     dict(angle=0,  scale=1.00, gain=1.0, bias=0),
    "rot_15":       dict(angle=15, scale=1.00, gain=1.0, bias=0),
    "rot_45":       dict(angle=45, scale=1.00, gain=1.0, bias=0),
    "scale_0.75":   dict(angle=0,  scale=0.75, gain=1.0, bias=0),
    "scale_0.50":   dict(angle=0,  scale=0.50, gain=1.0, bias=0),
    "light_gain":   dict(angle=0,  scale=1.00, gain=0.6, bias=0),
    "light_bias":   dict(angle=0,  scale=1.00, gain=1.0, bias=40),
}

pairs = {name: make_pair(base_gray, noise_sigma=2.0, **params)
         for name, params in DISTORTIONS.items()}

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.ravel()
axes[0].imshow(base_gray, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("original")
for ax, (name, (img, _)) in zip(axes[1:], pairs.items()):
    ax.imshow(img, cmap="gray", vmin=0, vmax=255)
    ax.set_title(name)
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Журнал экспериментов

Эксперимент — это воспроизводимая конфигурация, а не отдельный запуск. Каждая строка журнала должна позволять восстановить условия: изображение, искажение, детектор, его параметры, ограничение на число точек, допуск, seed.

In [ ]:
RUNS = []


def log_run(**fields):
    """Добавить запись в журнал экспериментов.

    Вход:  произвольные именованные поля (конфигурация + метрики + время).
    Выход: словарь добавленной записи (с полем run_id).
    """
    record = {"run_id": len(RUNS), **fields}
    RUNS.append(record)
    return record


def runs_table(columns=None, sort_by=None):
    """Вернуть журнал в виде таблицы pandas."""
    if not RUNS:
        return pd.DataFrame()
    frame = pd.DataFrame(RUNS)
    if columns:
        frame = frame[[c for c in columns if c in frame.columns]]
    if sort_by:
        frame = frame.sort_values(sort_by)
    return frame.reset_index(drop=True)


def save_runs(path=OUTPUT_DIR / "runs.csv"):
    """Сохранить журнал; путь указывается в отчёте."""
    runs_table().to_csv(path, index=False)
    return path

## 6. Детекторы и повторяемость

Типичная ошибка (см. [рубрику](../teachers-assessment/README.md#типичные-ошибки)): повторяемость измеряется на разных сценах, из-за чего сравнение несопоставимо. В этой заготовке базовое изображение одно, а меняется только искажение — сохраняйте это свойство при расширении серии.

Вторая ловушка — разное число точек у детекторов. Ограничивайте выдачу по силе отклика (`max_points`) одинаково для всех детекторов и фиксируйте это значение в журнале.

Ниже показан работающий пример на одном детекторе (FAST) и одном искажении. Это «рельсы»: убедитесь, что понимаете каждую строку, затем реализуйте единый интерфейс для остальных детекторов.

In [ ]:
def repeatability(pts1, pts2, H, shape, eps=3.0):
    """Повторяемость детектора на паре с известной гомографией.

    Вход:
        pts1, pts2 — (N, 2) и (M, 2), координаты (x, y);
        H          — (3, 3), отображает координаты первого кадра во второй;
        shape      — (H, W) второго кадра;
        eps        — допуск в пикселях.
    Выход:
        (rep, n_valid, n_matched): rep = n_matched / n_valid; в знаменателе —
        только точки, эталонное положение которых попало в кадр.
    """
    pts1 = np.asarray(pts1, dtype=np.float64).reshape(-1, 2)
    pts2 = np.asarray(pts2, dtype=np.float64).reshape(-1, 2)
    if len(pts1) == 0 or len(pts2) == 0:
        return 0.0, 0, 0
    projected = warp_points(H, pts1)
    h, w = shape[:2]
    inside = ((projected[:, 0] >= 0) & (projected[:, 0] < w) &
              (projected[:, 1] >= 0) & (projected[:, 1] < h))
    projected = projected[inside]
    if len(projected) == 0:
        return 0.0, 0, 0
    dist = np.linalg.norm(projected[:, None, :] - pts2[None, :, :], axis=2)
    n_matched = int((dist.min(axis=1) <= eps).sum())
    return n_matched / len(projected), int(len(projected)), n_matched


def draw_points(image, pts, title="", color=(0, 255, 0), radius=3):
    """Показать точки поверх изображения (служебная визуализация)."""
    canvas = cv2.cvtColor(image, cv2.COLOR_GRAY2BGR)
    for x, y in np.asarray(pts, dtype=np.float64).reshape(-1, 2):
        cv2.circle(canvas, (int(round(x)), int(round(y))), radius, color, 1, cv2.LINE_AA)
    plt.figure(figsize=(6, 6))
    plt.imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
    plt.title(f"{title} (точек: {len(pts)})")
    plt.axis("off")
    plt.show()

In [ ]:
# Рельсы: один детектор, одно искажение, один замер повторяемости.
MAX_POINTS = 400
EPS_PX = 3.0

fast = cv2.FastFeatureDetector_create(threshold=25, nonmaxSuppression=True)


def detect_fast(gray, max_points=MAX_POINTS):
    """Опорная реализация: FAST с отбором max_points сильнейших откликов."""
    keypoints = fast.detect(gray, None)
    keypoints = sorted(keypoints, key=lambda k: k.response, reverse=True)[:max_points]
    return np.array([k.pt for k in keypoints], dtype=np.float32).reshape(-1, 2)


img2, H12 = pairs["rot_15"]
p1 = detect_fast(base_gray)
p2 = detect_fast(img2)

t0 = time.perf_counter()
rep, n_valid, n_matched = repeatability(p1, p2, H12, img2.shape, eps=EPS_PX)
elapsed = time.perf_counter() - t0

log_run(image="astronaut", detector="fast", detector_params="threshold=25",
        distortion="rot_15", max_points=MAX_POINTS, eps_px=EPS_PX,
        n_kp1=len(p1), n_kp2=len(p2), n_valid=n_valid, n_matched=n_matched,
        repeatability=round(rep, 4), seconds=round(elapsed, 4), seed=SEED)

print(f"FAST / rot_15: повторяемость = {rep:.3f} ({n_matched} из {n_valid})")
draw_points(base_gray, p1, "FAST, исходный кадр")

In [ ]:
def detect_keypoints(gray, method, max_points=MAX_POINTS, **params):
    """Единый интерфейс детектора характеристических точек.

    Контракт.
    Вход:
        gray       — (H, W) uint8;
        method     — 'harris' | 'shi_tomasi' | 'fast' | 'dog';
        max_points — верхняя граница числа возвращаемых точек; отбор ведётся
                     по силе отклика, а не по порядку обхода изображения;
        params     — параметры конкретного детектора (k, blockSize, threshold,
                     qualityLevel, min_sigma и т. п.).
    Выход:
        np.ndarray (N, 2) float32, координаты в формате (x, y), N <= max_points.

    Подсказки по инструментам:
        harris      — cv2.cornerHarris + порог по отклику + немаксимальное подавление;
        shi_tomasi  — cv2.goodFeaturesToTrack;
        fast        — cv2.FastFeatureDetector_create (см. detect_fast выше);
        dog         — skimage.feature.blob_dog либо cv2.SIFT_create().detect.

    Требование: все параметры, влияющие на число точек, передавайте явно —
    они попадут в журнал и в отчёт.
    """
    raise NotImplementedError("TODO (задание 3.1): реализуйте единый интерфейс детекторов")

In [ ]:
# TODO (задание 3.1): постройте серию «детектор x искажение -> повторяемость».
#
# Требования к серии:
#   1) одно и то же базовое изображение для всех строк;
#   2) одинаковый max_points и одинаковый eps_px;
#   3) каждый запуск записывается через log_run со всеми параметрами;
#   4) отдельно проверьте чувствительность к eps_px (например, 2 и 5 пикселей) —
#      выводы не должны зависеть от одного произвольно выбранного допуска.
#
# DETECTORS = {
#     "harris":     dict(blockSize=3, ksize=3, k=0.04),
#     "shi_tomasi": dict(qualityLevel=0.01, minDistance=5),
#     "fast":       dict(threshold=25),
#     "dog":        dict(min_sigma=1.5, max_sigma=12, threshold=0.05),
# }
#
# for det_name, det_params in DETECTORS.items():
#     for dist_name, (img2, H12) in pairs.items():
#         ...
#         log_run(...)

runs_table()

## 7. Дескрипторы и фильтрация соответствий

SIFT создаётся через `cv2.SIFT_create()`, ORB — через `cv2.ORB_create()`. Модуль `cv2.xfeatures2d` в работе не используется.

Норма расстояния зависит от типа дескриптора: вещественный SIFT — `cv2.NORM_L2`, бинарный ORB — `cv2.NORM_HAMMING`. Смешение норм — частая причина «необъяснимо плохого» сопоставления.

Качество фильтрации оценивается не числом оставшихся соответствий, а их согласованностью с эталонной гомографией: ниже задана функция `match_precision`, которая считает долю соответствий, попадающих в допуск от эталонного положения.

In [ ]:
def build_feature(name):
    """Создать детектор-дескриптор и соответствующую норму расстояния.

    Выход: (feature, norm_type).
    """
    if name == "sift":
        return cv2.SIFT_create(), cv2.NORM_L2
    if name == "orb":
        return cv2.ORB_create(nfeatures=1000), cv2.NORM_HAMMING
    raise ValueError(f"Неизвестный дескриптор: {name}")


def knn_match(desc1, desc2, norm_type, k=2):
    """Сырое сопоставление полным перебором без кросс-проверки."""
    matcher = cv2.BFMatcher(norm_type, crossCheck=False)
    return matcher.knnMatch(desc1, desc2, k=k)


def match_precision(kp1, kp2, matches, H, eps=3.0):
    """Доля соответствий, согласованных с эталонной гомографией.

    Вход:  kp1, kp2 — списки cv2.KeyPoint; matches — список cv2.DMatch;
           H — эталонная гомография; eps — допуск в пикселях.
    Выход: (precision, n_correct, n_total).
    """
    if not matches:
        return 0.0, 0, 0
    src = np.array([kp1[m.queryIdx].pt for m in matches], dtype=np.float64)
    dst = np.array([kp2[m.trainIdx].pt for m in matches], dtype=np.float64)
    err = np.linalg.norm(warp_points(H, src) - dst, axis=1)
    n_correct = int((err <= eps).sum())
    return n_correct / len(matches), n_correct, len(matches)


def show_matches(img1, kp1, img2, kp2, matches, title="", max_draw=60):
    """Визуализация соответствий (служебная функция)."""
    shown = matches[:max_draw]
    canvas = cv2.drawMatches(img1, kp1, img2, kp2, shown, None,
                             flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
    plt.figure(figsize=(14, 6))
    plt.imshow(canvas, cmap="gray")
    plt.title(f"{title} — показано {len(shown)} из {len(matches)}")
    plt.axis("off")
    plt.show()

In [ ]:
# Рельсы: SIFT на паре с поворотом, сырые соответствия без фильтрации.
img2, H12 = pairs["rot_15"]
sift, sift_norm = build_feature("sift")

kp1, des1 = sift.detectAndCompute(base_gray, None)
kp2, des2 = sift.detectAndCompute(img2, None)
raw_knn = knn_match(des1, des2, sift_norm, k=2)
raw_matches = [pair[0] for pair in raw_knn if len(pair) > 0]

prec, n_ok, n_all = match_precision(kp1, kp2, raw_matches, H12, eps=EPS_PX)
print(f"SIFT, без фильтрации: {n_all} соответствий, доля верных {prec:.3f}")

log_run(image="astronaut", descriptor="sift", distortion="rot_15",
        filter="none", ratio=None, n_matches=n_all, n_correct=n_ok,
        precision=round(prec, 4), eps_px=EPS_PX, seed=SEED)

show_matches(base_gray, kp1, img2, kp2, raw_matches, "SIFT, без фильтрации")

In [ ]:
def filter_matches(kp1, kp2, knn_matches, method, ratio=0.75,
                   ransac_thresh=3.0):
    """Фильтрация соответствий.

    Контракт.
    Вход:
        kp1, kp2     — списки cv2.KeyPoint;
        knn_matches  — результат knn_match(..., k=2);
        method       — 'ratio' | 'cross' | 'ransac' | 'ratio+ransac';
        ratio        — порог ratio test;
        ransac_thresh— допуск RANSAC в пикселях.
    Выход:
        (matches, info), где matches — список cv2.DMatch, info — словарь;
        для схем с RANSAC info содержит ключи 'H_est' (3, 3) и 'inlier_ratio'.

    Требования:
        1) кросс-проверка реализуется явно (например, встречным сопоставлением
           или cv2.BFMatcher(..., crossCheck=True)), а не подменяется ratio test;
        2) при числе соответствий < 4 RANSAC не запускается — верните пустой
           список и зафиксируйте это в info['status'];
        3) функция не должна использовать эталонную гомографию H: она известна
           только для оценки качества, но не для фильтрации.
    """
    raise NotImplementedError("TODO (задание 3.2): реализуйте схемы фильтрации")

In [ ]:
def homography_error(H_est, H_true, shape, n_grid=8):
    """Средняя ошибка переноса точек регулярной сетки, пиксели.

    Вход:  H_est, H_true — (3, 3); shape — (H, W) исходного кадра.
    Выход: float (np.nan, если H_est не задана).
    """
    if H_est is None:
        return float("nan")
    h, w = shape[:2]
    xs, ys = np.meshgrid(np.linspace(0, w - 1, n_grid), np.linspace(0, h - 1, n_grid))
    pts = np.stack([xs.ravel(), ys.ravel()], axis=1)
    return float(np.mean(np.linalg.norm(warp_points(H_est, pts) - warp_points(H_true, pts), axis=1)))


# TODO (задание 3.2): исследуйте влияние порога ratio test и сравните схемы фильтрации.
#
# Минимальный план:
#   1) дескрипторы: sift и orb;
#   2) искажения: минимум поворот, масштаб и изменение освещённости;
#   3) схемы: 'ratio', 'cross', 'ransac', 'ratio+ransac' (не менее двух);
#   4) серия по ratio in [0.6, 0.7, 0.75, 0.8, 0.9] при прочих равных;
#   5) для каждого запуска логируйте: n_matches, precision, homography_error,
#      время; визуализируйте соответствия ДО и ПОСЛЕ фильтрации минимум
#      для одной пары.
#
# Ожидаемый эффект, который нужно проверить измерением: с ростом порога ratio
# число соответствий растёт, а доля верных падает.

runs_table()

## 8. Template matching против feature matching

`cv2.matchTemplate` сравнивает фиксированный шаблон со скользящим окном без нормировки по геометрии: метод не инвариантен ни к повороту, ни к масштабу. Ваша задача — не просто сообщить это, а измерить, при каком угле и масштабе метод перестаёт находить объект, и сопоставить границу отказа с поведением feature matching на тех же данных.

Метрика отказа задаётся заранее: ошибка положения найденного окна относительно эталонного положения центра шаблона превышает допуск (например, 10 пикселей).

In [ ]:
# Рельсы: шаблон вырезается из исходного кадра и ищется в неискажённом кадре.
TPL_BOX = (200, 180, 96, 96)  # x, y, w, h
tx, ty, tw, th = TPL_BOX
template = base_gray[ty:ty + th, tx:tx + tw]
template_center = np.array([[tx + tw / 2.0, ty + th / 2.0]])


def locate_template(scene, tpl, method=cv2.TM_CCOEFF_NORMED):
    """Найти шаблон в кадре.

    Выход: (center_xy (2,), score float).
    """
    response = cv2.matchTemplate(scene, tpl, method)
    _, max_val, _, max_loc = cv2.minMaxLoc(response)
    center = np.array([max_loc[0] + tpl.shape[1] / 2.0,
                       max_loc[1] + tpl.shape[0] / 2.0])
    return center, float(max_val)


center, score = locate_template(base_gray, template)
print("Неискажённый кадр: центр", center.round(1), "отклик", round(score, 3))

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].imshow(template, cmap="gray", vmin=0, vmax=255)
axes[0].set_title("шаблон")
canvas = cv2.cvtColor(base_gray, cv2.COLOR_GRAY2BGR)
cv2.rectangle(canvas, (tx, ty), (tx + tw, ty + th), (0, 255, 0), 2)
axes[1].imshow(cv2.cvtColor(canvas, cv2.COLOR_BGR2RGB))
axes[1].set_title("положение шаблона в исходном кадре")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# TODO (задание 3.3): найдите границу отказа template matching.
#
# План:
#   1) серия по углу поворота (0, 5, 10, 15, 30, 45 градусов) при scale = 1;
#   2) серия по масштабу (1.0, 0.9, 0.8, 0.7, 0.5) при angle = 0;
#   3) для каждой конфигурации: эталонное положение центра шаблона =
#      warp_points(H, template_center); ошибка положения = ||найденный - эталонный||;
#      отказ, если ошибка > FAIL_PX;
#   4) на тех же парах повторите поиск того же участка через feature matching
#      (например, оценив гомографию по SIFT + RANSAC и перенеся центр шаблона);
#   5) занесите обе ветви в журнал с полем method in {'template', 'features'}
#      и постройте общий график «искажение -> ошибка положения».
#
# FAIL_PX = 10.0
# for angle in [...]:
#     scene, H = make_pair(base_gray, angle=angle, noise_sigma=2.0)
#     expected = warp_points(H, template_center)[0]
#     ...
#     log_run(method="template", distortion=f"rot_{angle}", position_error=..., ok=...)

pass

## Отчёт

### Сводные таблицы

Обязательны две таблицы:

1. «детектор × искажение → повторяемость» (при фиксированных `max_points` и `eps_px`);
2. «дескриптор × схема фильтрации → число соответствий, доля верных, ошибка гомографии, время».

Таблицы строятся по журналу, а не набираются вручную.

In [ ]:
# Каркас сводных таблиц. Заполните журнал сериями выше — таблицы соберутся сами.
frame = runs_table()

if not frame.empty and {"detector", "distortion", "repeatability"} <= set(frame.columns):
    table_repeatability = frame.pivot_table(index="detector", columns="distortion",
                                            values="repeatability", aggfunc="mean")
    display(table_repeatability.round(3))
else:
    print("Таблица повторяемости пока пуста: выполните серию из раздела 6.")

if not frame.empty and {"descriptor", "filter"} <= set(frame.columns):
    cols = [c for c in ["n_matches", "precision", "homography_error", "seconds"]
            if c in frame.columns]
    table_matching = frame.pivot_table(index=["descriptor", "filter"], values=cols, aggfunc="mean")
    display(table_matching.round(3))
else:
    print("Таблица сопоставления пока пуста: выполните серию из раздела 7.")

save_runs()

### Выводы

Заполните три раздела раздельно. Смешение наблюдения и интерпретации — типичная причина потери баллов по критерию К3.

**Наблюдения** (только измеренные факты, со ссылкой на строки журнала):

-

**Интерпретация** (предполагаемые причины наблюдаемого):

-

**Выводы и их границы** (что именно проверено: какое изображение, какие диапазоны углов, масштабов, освещённости, какие значения `max_points` и `eps_px`; на что результат не переносится):

-

**Анализ ошибок**: приведите не менее двух конкретных случаев — пара, на которой детектор или схема фильтрации отказывает, с визуализацией и объяснением.

## Контрольные вопросы

Из [списка вопросов блока](README.md#контрольные-вопросы-блока):

1. Что такое повторяемость детектора точек и как её измерить?
2. Чем детектор Харриса отличается от DoG-детектора? Что делает SIFT инвариантным к масштабу?
3. Зачем нужен ratio test Лоу при сопоставлении дескрипторов?

Вопрос к защите: почему при увеличении порога ratio test растёт число соответствий и что происходит с их качеством? Ответ подкрепите своей серией, а не общими соображениями.

## Чек-лист перед сдачей

- [ ] Ноутбук исполняется сверху вниз без ошибок после `Restart & Run All`.
- [ ] Указаны ФИО, группа, номер работы, источники данных.
- [ ] Seed и версии библиотек зафиксированы и выведены.
- [ ] Повторяемость измерена на контролируемых искажениях одного изображения; `max_points` и `eps_px` одинаковы для всех детекторов.
- [ ] Проверена чувствительность выводов к допуску `eps_px`.
- [ ] Соответствия отфильтрованы минимум двумя способами, есть визуализация до и после фильтрации.
- [ ] Влияние порога ratio test исследовано серией, а не принято по умолчанию.
- [ ] Показан и измерен случай отказа template matching; проведено сравнение с feature matching на тех же парах.
- [ ] Есть сводные таблицы, журнал сохранён (`outputs_hw4/runs.csv`).
- [ ] Наблюдения, интерпретация и выводы разделены; указаны ограничения.
- [ ] Ответы на контрольные вопросы включены в отчёт.